In [1]:
import sys
import seaborn as sb
import pandas as pd
import numpy as np
from enum import Enum
from pathlib import Path
import matplotlib.pyplot as plt

In [2]:
current_dir = Path.cwd()
project_root = current_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print("Done")

Done


In [4]:
class Distributions(Enum):
    NORMAL = "normal"
    UNIFORM = "uniform"
    EXPONENTIAL = "exponential"
    CAUCHY = "cauchy"

In [7]:
K = int(input("Enter the no of distributions to fit: "))
dist_to_use = []
for i in range(K):
    dist_to_use.append(input(f"{i+1}th distribution to use: "))

In [8]:
dist_to_use

['normal', 'normal', 'exponential', 'cauchy']

In [9]:
def multivariate_normal(X: np.array, mean: np.array, cov: np.array) -> np.array:
    diff = X - mean
    num = np.exp(-0.5*(np.sum((diff @ np.linalg.inv(cov)) * diff, axis=1)))
    deno = np.sqrt((2*np.pi)**len(mean) * np.linalg.det(cov))
    return num/deno

In [10]:
no_of_features = 5
mean = np.random.randint(low=0, high=10, size=5).reshape(1, -1)
cov = np.eye(no_of_features)

In [11]:
no_of_rows = 10
X = np.random.uniform(size=(no_of_rows, no_of_features))

In [12]:
mean.shape, X.shape

((1, 5), (10, 5))

In [13]:
probs = multivariate_normal(X=X, mean=mean, cov=cov)

In [14]:
probs

array([1.71894282e-38, 7.66052889e-37, 2.16637659e-38, 2.63908226e-38,
       1.76464592e-40, 3.72194015e-35, 2.85135227e-35, 5.53722775e-36,
       4.75143711e-37, 7.53367552e-39])

### Exponential Distribution

In [15]:
x = np.array([[1,3,1,-1,5],
              [1,5,8,2,-2],
              [4,7,8,2,1]])

mask = x>0
np.where(mask, x, 0)

array([[1, 3, 1, 0, 5],
       [1, 5, 8, 2, 0],
       [4, 7, 8, 2, 1]])

In [16]:
def exponential_pdf(x, scale):
    scale = max(scale, 1e-15)  
    pdf_values = np.zeros_like(x, dtype=float)
    mask = x > 0
    pdf_values[mask] = (1.0 / scale) * np.exp(-x[mask] / scale)
    return pdf_values + 1e-15

In [17]:
def exponential_log_pdf(x, scale):
    scale = max(scale, 1e-15)
    log_pdf = -np.log(scale) - (x / scale)
    log_pdf[x <= 0] = -np.inf
    return log_pdf

In [18]:
def multivariate_exponential_log(X: np.array, scale_vec: np.array) -> np.array:
    log_probs = np.zeros((X.shape[0], 1))
    
    for i in range(X.shape[1]):
        col_log_pdf = exponential_log_pdf(X[:, i], scale_vec[i].item())
        log_probs += col_log_pdf.reshape(-1, 1)
        
    return log_probs

In [19]:
no_of_features = 5
scales = np.random.randint(low=1, high=10, size=no_of_features).reshape((1, no_of_features))
X = np.random.uniform(size=(10, no_of_features))

In [20]:
X.shape, scales.shape

((10, 5), (1, 5))

In [21]:
multivariate_exponential_log(X=X, scale_vec=scales.flatten())

array([[-7.16788664],
       [-7.01965998],
       [-7.02344285],
       [-7.11514385],
       [-7.19415277],
       [-6.7943124 ],
       [-7.07665445],
       [-7.2227067 ],
       [-7.00550489],
       [-7.10239747]])

In [24]:
x = np.array([[1, 3, 5],
              [0.4, 0.2, 0.1],
              [2, 0.9, 0.8]])

mask = ((x<1)&(x>0))
mask

array([[False, False, False],
       [ True,  True,  True],
       [False,  True,  True]])

In [76]:
def uniform_distribution(X: np.array, low: float = 0, high: float = 1) -> np.array:
    """given 1d x follows unifrom returns probs

    Args:
        X (np.array): data
        low (float): low parameter
        high (float): high parameter

    Returns:
        np.array: probs
    """
    mask = ((X <= high) & (X >= low))
    return (np.where(mask, -np.log(high - low + 1e-15), -np.inf)).reshape((1, len(X)))

In [77]:
x = np.array([2, 1, 0.6, 0.4, 0.7, 0.8, 0.11])
probs = uniform_distribution(X=x, low=2, high=5)

In [78]:
probs

array([[-1.09861229,        -inf,        -inf,        -inf,        -inf,
               -inf,        -inf]])

In [82]:
def multivariate_uniform_dist(X: np.array, low: np.array, high: np.array) -> np.array:
    """given multidim x follows unifrom returns probs

    Args:
        X (np.array): data
        low (np.array): low vector
        high (np.array): high vector

    Returns:
        np.array: probs
    """
    probs = np.zeros(shape=(1, X.shape[0]))
    for i in range(X.shape[1]):
        probs += uniform_distribution(X[:, i].flatten(), low=low[i], high=high[i])
    
    return probs

In [85]:
x = np.random.standard_normal(size=(10, 4))

low = np.array([2, 6, 7, 1])
high = np.array([4, 8, 9, 4])

probs = multivariate_uniform_dist(X=x, low=low, high=high)

In [86]:
probs

array([[-inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf]])

In [91]:
def cauchy_distribution(X: np.array, loc: float, scale: float) -> np.array:
    """if x follows cauchy this function returns probs

    Args:
        X (np.array): data
        loc (float): location parameter
        scale (float): scale parameter

    Returns:
        np.array: probs
    """
    prob = -np.log(np.pi) -np.log(scale) - np.log(1 + ((X - loc)/scale)**2)
    return prob     

In [92]:
x = np.array([2, 1, 0.6, 0.4, 0.7, 0.8, 0.11])
probs = cauchy_distribution(X=x, loc=2, scale=5)
probs

array([-2.7541678 , -2.79338851, -2.82964626, -2.85165742, -2.81958094,
       -2.81016999, -2.88772269])

In [93]:
def multivariate_cauchy_dist(X:np.array, loc: np.array, scale: np.array) -> np.array:
    """if x follows multivariate cauchy dist this function returns probs

    Args:
        X (np.array): data 
        loc (np.array): location vectir parameter
        scale (np.array): scale vector parameter

    Returns:
        np.array: probs
    """
    probs = np.zeros((1, X.shape[0]))
    for i in range(X.shape[1]):
        probs += cauchy_distribution(X=X[:, i].flatten(), loc=loc[i], scale=scale[i])
        
    return probs

In [94]:
x = np.random.standard_normal(size=(10, 4))

loc = np.array([2, 6, 7, 1])
scale = np.array([4, 8, 9, 4])

probs = multivariate_cauchy_dist(X=x, loc=loc, scale=scale)
probs

array([[-13.16488928, -12.9915282 , -13.39139786, -13.2606439 ,
        -12.6201592 , -12.87425562, -12.89701501, -12.91184029,
        -12.87217695, -13.21464818]])